# Question 14 - 134. Gas Station

There are `n` gas stations along a **circular** route, where the amount of gas at the `i-th` station is `gas[i]`.

You have a car with an unlimited gas tank and it costs `cost[i]` of gas to travel from the `i-th` station to its next `(i + 1)-th` station. You begin the journey with an empty tank at one of the gas stations.

Given two integer arrays `gas` and `cost`, return *the starting gas station's index if you can travel around the circuit once in the clockwise direction, otherwise return* `-1`. If there exists a solution, it is **guaranteed** to be **unique**.

**Example 1:**

    Input: gas = [1,2,3,4,5], cost = [3,4,5,1,2]
    Output: 3
    Explanation:
    Start at station 3 (index 3) and fill up with 4 unit of gas. Your tank = 0 + 4 = 4
    Travel to station 4. Your tank = 4 - 1 + 5 = 8
    Travel to station 0. Your tank = 8 - 2 + 1 = 7
    Travel to station 1. Your tank = 7 - 3 + 2 = 6
    Travel to station 2. Your tank = 6 - 4 + 3 = 5
    Travel to station 3. The cost is 5. Your gas is just enough to travel back to station 3.
    Therefore, return 3 as the starting index.

**Example 2:**

    Input: gas = [2,3,4], cost = [3,4,3]
    Output: -1
    Explanation:
    You can't start at station 0 or 1, as there is not enough gas to travel to the next station.
    Let's start at station 2 and fill up with 4 unit of gas. Your tank = 0 + 4 = 4
    Travel to station 0. Your tank = 4 - 3 + 2 = 3
    Travel to station 1. Your tank = 3 - 3 + 3 = 3
    You cannot travel back to station 2, as it requires 4 unit of gas but you only have 3.
    Therefore, you can't travel around the circuit once no matter where you start.

**Constraints:**

- `n == gas.length == cost.length`
- `1 <= n <= 10^5`
- `0 <= gas[i], cost[i] <= 10^4`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy: running tank with restart, plus a total-balance check

📘 **Revise first:** [Pattern C · running best-so-far](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (simulate from every start) | O(n²) | O(1) |
| 2️⃣ Optimized (one pass with restart) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Gas stations sit on a **circular** road. At station `i` you can fill up `gas[i]`, and driving to the next station burns `cost[i]`. You start with an **empty** tank. Which station should you start from so you can drive **all the way round once** without running out? If none works, answer `-1`. (If an answer exists, it's unique.)

**First simplification:** what matters at each station is the **net** effect, `gain[i] = gas[i] − cost[i]`. Positive means you leave with more fuel than you arrived with; negative means the next leg drains you.

```
gas  = [1, 2, 3, 4, 5]
cost = [3, 4, 5, 1, 2]
gain = [-2, -2, -2, +3, +3]
```

**Everyday picture: a road trip where your wallet can't go negative.** Each stop gives or takes money. Two common-sense facts:
1. **If the total of all gains is negative, no starting point can work.** You can't finish a loop that loses money overall.
2. **If you start somewhere and go broke at stop `j`, then starting at any stop in between would also go broke by `j`.** When you passed those stops you had **zero or more** money in hand, and starting fresh there would give you exactly zero, which is no better. So skip straight to `j + 1`.

Those two facts give a **one-pass** solution.

### Step 0 · Clarify before coding

🗣️ *"At each station I gain gas[i] and spend cost[i] to reach the next, so I'll think in terms of the net gain per station. The route is circular, I start with an empty tank, and if a valid start exists it's unique."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Circular route? | **Yes**: after the last station comes station 0. | |
| Is a tank of exactly 0 on arrival OK? | **Yes**; only *negative* fails. | Use `< 0` as the failure test. |
| Unique answer if one exists? | **Yes.** | Just find it; no need to compare candidates. |
| Size? | Up to 10⁵ stations. | O(n²) = 10¹⁰, too slow. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** try every station as the start and **simulate** the full loop, giving up on that start as soon as the tank goes negative.

🗣️ *"The direct approach simulates the trip from every starting station. That's n starts times up to n steps, O(n²). Too slow at 10⁵, but simulating shows me what a failure tells me."*

In [1]:
class SolutionBrute:
    def canCompleteCircuit(self, gas: list[int], cost: list[int]) -> int:
        n = len(gas)
        for start in range(n):
            tank = 0
            for k in range(n):
                i = (start + k) % n            # wrap around the circle
                tank += gas[i] - cost[i]
                if tank < 0:
                    break                      # this start fails
            else:                              # loop finished without a break
                return start
        return -1

### Step 2 · Optimize: the "aha" moment

Brute force wastes work: after start `s` fails at station `j`, it tries `s + 1`, `s + 2`, … and re-drives the same stretch.

> 💡 **Aha #1: one total decides whether an answer exists.** If `sum(gain) < 0` → `-1`. If `sum(gain) ≥ 0` → a valid start is **guaranteed**.
>
> 💡 **Aha #2: a failure rules out the whole stretch.** If starting at `s` you first go negative leaving station `j`, then **no station from `s` to `j` can be the start.** Each of them was reached with a tank **≥ 0**; starting there means a tank of exactly 0, which can only do worse. So the next candidate is `j + 1`.

**One pass:**
- Keep `tank` (fuel since the current candidate start) and `total` (all gains).
- If `tank` drops below 0 at station `i` → candidate becomes `i + 1`, reset `tank = 0`.
- At the end: if `total ≥ 0`, the last candidate is the answer; otherwise `-1`.

**But we never drove the stations *before* the final start: how do we know they're fine?** Because the total is ≥ 0. The stretch from `start` to the end has a surplus, and the stretch before `start` has a deficit; together they add up to the total, which is ≥ 0. So the surplus is big enough to cover the deficit when we wrap around.

🗣️ *"First, if total gas is less than total cost, it's impossible. Otherwise a start exists. I scan once with a running tank. If the tank goes negative at station i, then no station from my current candidate through i can work, because each of them would arrive at i with no more gas than I had. So I move the candidate to i + 1 and reset the tank. The final candidate is the answer. One pass, O(n) time, O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Simulate from every start | ❌ | O(n²). Re-drives stretches already proven bad. |
| Double the array and slide a window | ⚠️ | Works, but uses O(n) extra memory and fiddly indexing for no gain. |
| Start right after the **lowest point** of the running total | ✅ (equivalent) | The station after your "most broke" moment is the right start. Same O(n)/O(1). A nice alternative to mention. |
| **Greedy restart + total check** | ✅ **best** | One pass, three variables, and it's backed by two short proofs. |

In [2]:
class Solution:
    def canCompleteCircuit(self, gas: list[int], cost: list[int]) -> int:
        total = 0      # sum of all gains: decides if ANY answer exists
        tank = 0       # fuel since the current candidate start
        start = 0
        for i in range(len(gas)):
            gain = gas[i] - cost[i]
            total += gain
            tank += gain
            if tank < 0:            # can't leave station i from `start`
                start = i + 1       # every station in [start, i] is ruled out
                tank = 0
        return start if total >= 0 else -1

### Step 3 · Toy example walkthrough (tell it like a story)

`gas = [1, 2, 3, 4, 5]`, `cost = [3, 4, 5, 1, 2]` → `gain = [−2, −2, −2, +3, +3]`

| station i | gain | tank after | tank < 0? | candidate start | total |
|---|---|---|---|---|---|
| 0 | −2 | −2 | ✅ restart | 1 | −2 |
| 1 | −2 | −2 | ✅ restart | 2 | −4 |
| 2 | −2 | −2 | ✅ restart | 3 | −6 |
| 3 | +3 | 3 | ❌ | 3 | −3 |
| 4 | +3 | 6 | ❌ | 3 | **0** |

`total = 0 ≥ 0` → answer **3** ✅

**Narrated:** "Starting at 0, I'm short by 2 immediately, so 0 is out, and I try 1. Short again, try 2. Short again, try 3. From 3 I gain 3, then 3 more, so I'm at 6. The overall total is 0, which isn't negative, so a loop is possible, and 3 is the start. The 6 I banked from stations 3 and 4 exactly covers the 6 I'd lose going through 0, 1, 2."

**Impossible case:** `gas = [2, 3, 4]`, `cost = [3, 4, 3]` → gains `[−1, −1, +1]`, total = −1 < 0 → **−1** ✅

In [3]:
cases = [
    ([1, 2, 3, 4, 5], [3, 4, 5, 1, 2], 3),
    ([2, 3, 4], [3, 4, 3], -1),
    ([5], [4], 0),
    ([4], [5], -1),
    ([3, 1, 1], [1, 2, 2], 0),
    ([1, 2], [2, 1], 1),
]
for gas, cost, expected in cases:
    assert SolutionBrute().canCompleteCircuit(gas, cost) == expected
    assert Solution().canCompleteCircuit(gas, cost) == expected
    print(gas, cost, "->", expected)

import random
for _ in range(1000):
    n = random.randint(1, 8)
    gas = [random.randint(0, 5) for _ in range(n)]
    cost = [random.randint(0, 5) for _ in range(n)]
    b = SolutionBrute().canCompleteCircuit(gas, cost)
    o = Solution().canCompleteCircuit(gas, cost)
    assert (b == -1) == (o == -1), (gas, cost)          # agree on feasibility
    if o != -1:                                          # and the returned start really works
        assert SolutionBrute().canCompleteCircuit(gas[o:] + gas[:o], cost[o:] + cost[:o]) == 0
print("All tests passed ✅")

[1, 2, 3, 4, 5] [3, 4, 5, 1, 2] -> 3
[2, 3, 4] [3, 4, 3] -> -1
[5] [4] -> 0
[4] [5] -> -1
[3, 1, 1] [1, 2, 2] -> 0
[1, 2] [2, 1] -> 1
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Simulate every start | O(n²) | O(1) | n starts × up to n stations each. |
| **Greedy restart** | **O(n)** | **O(1)** | Each station is visited once; failed stretches are skipped whole. |

**Why O(n) time?** A single loop over the stations with a few additions and one comparison each.

**Why O(1) memory?** Just `total`, `tank`, and `start`.

**The two facts to be ready to prove:**
1. `total ≥ 0` ⇔ some start works.
2. Running out at `j` from `s` ⇒ every start between `s` and `j` also runs out (they'd reach `j` with no more fuel than you had).

**Edge cases to mention:** a single station, all zeros (start 0), a station where `gas == cost`, the only deficit at the very end.

**Likely follow-up:** *"What if answers weren't unique and you wanted all of them?"* → Every station right after a **lowest point** of the running total is a valid start.

---

#### 🗣️ Full interview script (about 60 seconds)

*"I'll turn each station into a net gain: gas minus cost.*

*Brute force simulates the loop from every start, which is O(n²).*

*Two observations make it linear. First, if the total gain is negative, no start can work; if it's non-negative, a start is guaranteed. Second, if I start at s and the tank goes negative leaving station j, then no station between s and j can be the start, because I reached each of them with a non-negative tank, and starting there fresh gives a tank of zero, which is no better.*

*So I scan once with a running tank. Whenever it goes negative, I move my candidate start to the next station and reset the tank. At the end, if the total is non-negative, the candidate is the answer; otherwise −1. O(n) time, O(1) space."*